# SpecMod tutorial

Fitting a source model to the spectra of a small induced earthquake, end to end.

The event is the Preston New Road **Mw 1.6** of 2019-08-26, recorded on the
LV and UR networks during hydraulic fracturing near Blackpool, UK. The
waveforms and station metadata are committed with the package (224 KB), so
this notebook needs no network access.

Five stages, one section each:

1. **Read and prepare** the waveforms — geometry, picks, instrument response.
2. **Cut** a signal window and a noise window to judge it against.
3. **Transform** both to spectra and select the usable bandwidth.
4. **Fit** a source model over that bandwidth.
5. **Save** the results.

Every processing step is written down with its equation in
[`docs/processing.md`](../docs/processing.md); this is the same pipeline with
the numbers left in.

## 1. Read and prepare

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
from obspy import UTCDateTime, read, read_inventory

import specmod.preprocess as pre
import specmod.utils as ut

# Relative to this notebook, so it runs from wherever it is opened.
DATA = Path("Data/2019-08-26T07:30:47.0")
METADATA = Path("MetaData")
OUTPUT = Path("Spectra")

The origin is what distances and theoretical arrivals are measured from.

Note the origin time is 18 minutes *after* the waveform start — the records
here begin at 07:30:47 and the catalogue origin is 07:49:24. Nothing in this
notebook depends on that offset, because every window is cut relative to the
**picks** rather than to the origin. It matters in one place only, and
`preprocess` now warns rather than producing nonsense: see
`set_picks_from_pyrocko`, which will not extrapolate a missing S arrival when
the origin does not precede the P pick.

In [ ]:
olat, olon, odep = 53.784, -2.967, 2.1
otime = UTCDateTime("2019-08-26T07:49:24.2")

In [ ]:
# The two horizontal components. S-wave amplitudes are what the source model
# is fitted to, so the verticals are not read.
st = read(str(DATA / "*HHE*"), format="mseed") + read(
    str(DATA / "*HHN*"), format="mseed"
)
inv = read_inventory(str(METADATA / "pnr_inventory.xml"), "stationxml")
print(f"{len(st)} traces, {len({tr.stats.station for tr in st})} stations")

In [ ]:
# Source-receiver geometry, from the inventory. Sets repi, rhyp, azimuth and
# back-azimuth on every trace.
pre.set_stream_distance(st, olat, olon, odep, otime, inventory=inv, dtype="mseed")

# P and S arrivals, from a Snuffler marker file.
pre.set_picks_from_pyrocko(st, str(next(DATA.glob("*.picks"))))

# A trace with no S pick cannot have an S window cut from it. Dropping them
# here is the pipeline's own idiom for "unusable".
for tr in st:
    if "s_time" not in tr.stats:
        st.remove(tr)

print(f"{len(st)} traces with both picks")

**Instrument correction happens here, in the notebook, not inside the
package.** That is deliberate: the choice of output units and water level is
part of the science, and burying it in a library call makes it invisible in
the record of what was run.

Demeaning matters more than it looks. A non-zero mean puts all of its energy
in the DC bin, which every estimator discards — so leaving it in loses energy
that the Parseval check would report as a failure.

In [ ]:
st.detrend("linear")
st.detrend("demean")
st.taper(0.05)
st.remove_response(inv, output="VEL")   # ground velocity, m/s

In [ ]:
ut.plot_traces(st.copy(), plot_theoreticals=True, conv=1)
plt.show()

## 2. Cut the windows

The S-window opens at a fixed fraction of the elapsed P–S time after the P
arrival, then is **refined** onto the part of it that actually carries energy:
the window is tightened to where the cumulative squared amplitude runs between
its 1st and 99th percentiles. That is why these come out at 1.8–3.7 s rather
than the nominal 20 s.

The noise window ends shortly before the P arrival and is *asked for* the same
length as the refined signal window. It rarely gets it — see below.

In [ ]:
sig = pre.get_signal(
    st, pre.cut_s, rafp=0.8, tafs=20, time_after="absolute_time", refine_window=True
)
noise = pre.get_noise_p(st, sig)

In [ ]:
# Every noise window here is shorter than the signal it is judged against,
# because the records begin only ~2 s before the P arrival. This is normal and
# is corrected for; it is why `SpectrumPair` carries a resolution floor rather
# than assuming the two spectra share a frequency resolution.
for s, n in list(zip(sig, noise))[:4]:
    asked = float(n.stats["wend_requested"] - n.stats["wstart_requested"])
    got = float(n.stats["wend"] - n.stats["wstart"])
    print(
        f"{s.id:16s} signal {s.stats.endtime - s.stats.starttime:5.2f} s   "
        f"noise asked {asked:5.2f} s, got {got:5.2f} s"
    )

In [ ]:
ut.plot_traces(st.copy(), plot_windows=True, conv=1, sig=sig, noise=noise)
plt.show()

## 3. Spectra and bandwidth

`spectrum_set_from_streams` transforms both windows, puts the noise on the
signal's frequency axis, bins both, raises the noise to account for what sits
*under* the signal, takes the ratio and selects the band where it passes.

Which estimator is used is configuration, not code: `fft`, `welch`,
`multitaper`, `quadratic` and `cwt` all satisfy the same Parseval contract, so
they are interchangeable here. The shipped default is `multitaper`.

In [ ]:
from specmod.pipeline import spectrum_set_from_streams

spectra = spectrum_set_from_streams(sig, noise)
print(f"{len(spectra)} spectra for event {spectra.event}")
print(f"{sum(p.passes for p in spectra.pairs.values())} passed the signal-to-noise gate")

In [ ]:
# One station in detail: signal, noise, the binned spectra the ratio is
# actually computed on, the selected band (red) and the resolution floor (grey).
from specmod.plotting import plot_pair, plot_set

station = spectra.ids()[0]
plot_pair(spectra[station], id=station, show_binned=True)
plt.show()

In [ ]:
pair = spectra[station]
print(f"band            {pair.band[0]:.2f} to {pair.band[1]:.2f} Hz")
print(f"resolution floor {pair.resolution_floor:.2f} Hz  (the shorter window's 1/T)")
print(f"units            {pair.signal.unit}")

### Changing ground-motion domain

The response was removed to velocity, but a source model is often read on
displacement. `to_motion` converts the whole event and **returns a new set** —
the velocity one is untouched, so both exist at once.

In [ ]:
displacement = spectra.to_motion("displacement")
print(f"velocity     {spectra[station].signal.unit}")
print(f"displacement {displacement[station].signal.unit}")

# The unbinned signal-to-noise ratio is invariant under this — both spectra are
# divided by the same 2*pi*f — but the *binned* ratio is not, because a bin
# holds a geometric mean. So a few bands do move.
moved = [i for i in spectra.ids() if spectra[i].band != displacement[i].band]
print(f"bands that moved: {len(moved)} of {len(spectra)}")

## 4. Fit a source model

The model comes from configuration — a Brune source with constant Q by
default — and the initial guesses are derived from each spectrum: the plateau
from the largest amplitude inside the band, the corner from where that
maximum falls.

`FitSpectra(spectra)` then `fit_spectra()` is the whole thing. The minimiser
(Powell), the `t*` lower bound and whether to fit the binned or unbinned
spectrum all come from `[fitting]` in the configuration, so the defaults are
recorded rather than remembered.

In [ ]:
from specmod import sources
from specmod.fitting import FitSpectra, initial_guess

print(sources.from_config().describe())
guess = initial_guess(spectra)
print(f"guesses for {len(guess)} of {len(spectra)} stations")

In [ ]:
fits = FitSpectra(spectra)
fits.fit_spectra()
print(f"{len(fits.models)} fitted, {fits.table['pass_fitting'].sum()} passed the fit checks")

The guess is only a starting point, and a crude one: it takes the largest
amplitude inside the band, which on a velocity spectrum is *near* the corner
but can land at the band edge when the corner sits outside the resolvable
range. Comparing it with where the fit ended up shows how much work the
minimiser is doing.

In [ ]:
import pandas as pd

fitted = fits.table.set_index("id")["fc"]
comparison = pd.DataFrame(
    {
        "guessed fc": {k: v["fc"] for k, v in guess.items()},
        "fitted fc": fitted,
        "band high": {k: spectra[k].band[1] for k in guess},
    }
).round(2)
comparison.head(8)

In [ ]:
# The fitted model over the spectrum it was fitted to.
plot_pair(spectra[station], id=station, fit=fits.models[station])
plt.show()

In [ ]:
# Or the whole event at once.
plot_set(spectra, fits=fits, columns=4)
plt.show()

In [ ]:
fits.table[["id", "fc", "fc-stderr", "llpsp", "ts", "pass_fitting"]].head(10)

**`fc-stderr` is empty, and that is the minimiser's doing rather than a
fault.** Powell — the shipped default, and what the published workflow used —
searches without building a covariance matrix, so lmfit has no uncertainties
to report. If you need error bars, fit with a method that estimates them:

```python
fits.fit_spectra(method="leastsq")
```

That costs something. On these 28 windows `leastsq` and Powell reach the same
median goodness-of-fit but disagree by more than 1% on 5 stations, which is
the signature of a shallow misfit surface with several local minima rather
than of one method being right. Choose deliberately, and record the choice —
`[fitting] method` in the configuration is where it belongs.

`pass_fitting` marks a fit whose parameter is pinned against one of its
bounds: the minimiser saying "further, if you would let me", with the bound
reported instead of a measurement. Without uncertainties the test is weaker,
because it can only ask whether the value *is* the bound rather than whether
it reaches one.

## 5. Save the results

Two formats, because the data is used two different ways.

**Spectra go to HDF5** — one file per event, one group per channel, with the
units, duration and sampling rate stored as attributes rather than assumed.
Nothing in the file names a Python class, which is the whole point: the
previous format was pickle, and a pickle stops loading the moment a class is
renamed.

**Fit tables go to Parquet**, which keeps its dtypes and can be queried by
DuckDB or polars without being loaded. CSV is still written when you ask for
it, because journal supplements want one.

In [ ]:
from specmod.io import load, save

path = save(OUTPUT / spectra.event, spectra)
print(f"{path}  ({path.stat().st_size // 1024} KB)")

back = load(path)
print(f"reloaded {len(back)} spectra; band unchanged: "
      f"{back[station].band == spectra[station].band}")

In [ ]:
FitSpectra.write_flatfile(OUTPUT / "FlatFiles" / f"{spectra.event}.parquet", fits)
FitSpectra.write_flatfile(OUTPUT / "FlatFiles" / f"{spectra.event}.csv", fits)
sorted(p.name for p in (OUTPUT / "FlatFiles").iterdir())

---

## Where to go next

- [`docs/processing.md`](../docs/processing.md) — every stage above with its
  equation and a pointer to the code that applies it.
- `specmod.config` — the layered configuration. A study pins its values in a
  committed TOML file; `specmod config show` prints what a run resolved to.
- `specmod.transforms` — the five spectral estimators and the Parseval
  contract they share.
- `specmod.sources` — Brune and Boatwright sources, constant and
  frequency-dependent Q.